### Setting up

In [ ]:
# Import packages
import uuid
import pandas as pd
import numpy as np
from os import environ
from fabric_api import Auth, Workspace, Dataset, Report, Dataflow, Capacity, Admin, Operations, KQLDatabase, Database, Pipeline
from dotenv import load_dotenv

# Load .env only if any credential is missing from the environment
_required_vars = ['TENANT_ID', 'CLIENT_ID', 'CLIENT_SECRET']
if not all(environ.get(v) for v in _required_vars):
    load_dotenv('./utils/.env')

# Tenant/app settings
TENANT_ID = environ.get('TENANT_ID', '')
CLIENT_ID = environ.get('CLIENT_ID', '')
CLIENT_SECRET = environ.get('CLIENT_SECRET', '')
FABRIC_SQL_ENDPOINT = environ.get('FABRIC_SQL_ENDPOINT', '')
FABRIC_DATABASE = environ.get('FABRIC_DATABASE', '')

In [ ]:
# Authentication (get bearer token)
auth = Auth(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
pbi_token = auth.get_token('pbi')
fabric_token = auth.get_token('fabric')

# Initializing objects
workspace = Workspace(pbi_token)
dataset = Dataset(pbi_token)
report = Report(pbi_token)
dataflow = Dataflow(pbi_token)
capacity = Capacity(pbi_token)
admin = Admin(pbi_token)
operations = Operations(fabric_token)
pipeline = Pipeline(fabric_token)
db = Database(FABRIC_SQL_ENDPOINT, FABRIC_DATABASE, CLIENT_ID, CLIENT_SECRET)

### Getting workspaces data

In [ ]:
# Get workspaces list
workspaces = workspace.list_workspaces()
workspaces_list = workspaces.get('content', [])

# See content
df_workspaces = pd.DataFrame(workspaces_list)
df_workspaces.sort_values(by='name', inplace=True)
df_workspaces.reset_index(inplace=True, drop=True)
n = len(df_workspaces.index)
print(f'Found {n} workspaces...\nPrinting:')
df_workspaces.head(n)

### List all users with access to a dataset

In [ ]:
#ataset_users = dataset.list_users()
try:
    workspace_to_search = 'Test workspace'
    workspace_to_search_id = df_workspaces['id'][df_workspaces['name']==workspace_to_search].values[0]
except IndexError:
    workspace_to_search_id = ''
print(f'Workspace ID: {workspace_to_search_id}')

dataset_to_search = 'Test dataset'

if workspace_to_search_id != '':
    datasets_list = dataset.list_datasets(workspace_id=workspace_to_search_id)
    df_datasets = pd.DataFrame(datasets_list['content'])
    dataset_to_search_id = df_datasets['id'][df_datasets['name']==dataset_to_search].values[0]
else:
    dataset_to_search_id = ''
print(f'Dataset ID: {dataset_to_search_id}')

if dataset_to_search_id != '':
    dataset_users = dataset.list_users(workspace_id=workspace_to_search_id, dataset_id=dataset_to_search_id)
    df_dataset_users = pd.DataFrame(dataset_users['content'])
    df_dataset_users['workspace'] = workspace_to_search
    df_dataset_users['report'] = dataset_to_search

    df_dataset_users.to_excel(f'./data/users/{dataset_to_search}_users.xlsx', index=False)

### Execute a DAX query against a dataset

In [ ]:
# Execute a DAX query (includes COUNTROWS pre-check for truncation detection)
workspace_id = 'your-workspace-id'
dataset_id = 'your-dataset-id'
query = "EVALUATE SUMMARIZECOLUMNS('Table'[Column1], 'Table'[Column2], 'Table'[Column3])"

result = dataset.execute_query(workspace_id=workspace_id, dataset_id=dataset_id, query=query)

if result['message'] == 'Success':
    print(f"Rows returned: {result['rows_returned']}")
    print(f"Total rows in source: {result['total_rows']}")
    print(f"Max rows allowed: {result['max_rows_allowed']} (based on {result['num_columns']} columns)")
    print(f"Truncated: {result['truncated']}")

    df_query = pd.DataFrame(result['content'])
    df_query.head()
else:
    print(f"Error: {result}")

### Add user to all workspaces

In [ ]:
users_to_be_added = [    
    {
        'user': 'test@test.com',
        'role': 'Member'
    }
]

for user_to_add in users_to_be_added:

    for workspace_data in workspaces_list:
        workspace_id = workspace_data['id']
        workspace_name = workspace_data['name']
        try:
            response = workspace.add_user(user_principal_name=user_to_add['user'], access_right=user_to_add['role'], workspace_id=workspace_id)
            if response['message'] != 'Success':
                response = workspace.update_user(user_principal_name=user_to_add['user'], access_right=user_to_add['role'], workspace_id=workspace_id)
        except Exception as e:
            print(f'Error:\n\nworkspace_id{workspace_id}\nworkspace_name={workspace_name}\nError message:\n{e}')

### Remove user from all workspaces

In [ ]:
users_to_be_removed = [
    'test@test.com'
]

for user_to_remove in users_to_be_removed:
    for workspace_data in workspaces_list:
        workspace_id = workspace_data['id']
        workspace_name = workspace_data['name']
        try:
            response = workspace.remove_user(user_principal_name=user_to_remove, workspace_id=workspace_id)
            print(f"Removed {user_to_remove} from {workspace_data['name']}")
        except Exception as e:
            print(f'Error: workspace_id{workspace_id} - workspace_name={workspace_name} - Error message: {e}')

### Copy a Dataflow to Another Workspace

In [ ]:
source_workspace_id = '123'
destination_workspace_id = '456'
dataflows_list_raw = dataflow.list_dataflows(workspace_id=source_workspace_id)
dataflows_list = dataflows_list_raw['content']

dataflows_to_copy_list = ['dataflow1', 'dataflow2']
dataflows_to_copy = [d for d in dataflows_list if d.get('name', 'not found') in dataflows_to_copy_list]

In [ ]:
import requests 
for d in dataflows_to_copy:
    dataflow_id = d['objectId']
    dataflow_name = d['name']

    dataflow_content = dataflow.get_dataflow_details(workspace_id=source_workspace_id, dataflow_id=dataflow_id).get('content', '')
    
    if dataflow_content != '':
        
        dataflow_content['entities'][0].pop('partitions', None)
        dataflow_content['pbi:mashup']['allowNativeQueries'] = False

        r = dataflow.create_dataflow(workspace_id=destination_workspace_id, dataflow_content=dataflow_content)
        print(r)

### Pipeline - Get Activities

In [ ]:
# Get pipeline definition
workspace_id = 'your-workspace-id'
pipeline_id = 'your-pipeline-id'

result = pipeline.get_pipeline_definition(workspace_id=workspace_id, pipeline_id=pipeline_id)
result

In [ ]:
# Get pipeline activities
result = pipeline.get_pipeline_activities(workspace_id=workspace_id, pipeline_id=pipeline_id)

if result['message'] == 'Success':
    activities = result['content']
    df_activities = pd.DataFrame(activities)
    print(f'Found {len(activities)} activities')
    df_activities[['name', 'type']].head(20)
else:
    print(f"Error: {result['message']}")

### Pipeline - Find Pipelines by Dataflow

In [ ]:
# Find all pipelines that reference a specific dataflow
workspace_id = 'your-workspace-id'
dataflow_id = 'your-dataflow-id'

result = pipeline.find_pipelines_by_dataflow(workspace_id=workspace_id, dataflow_id=dataflow_id)

if result['message'] == 'Success':
    for match in result['content']:
        print(f"Pipeline: {match['pipeline_name']} ({match['pipeline_id']})")
        print(f"  Activities: {', '.join(match['activities'])}")
else:
    print(f"Error: {result['message']}")

### Dataflow - Change Data Destination & Update Pipelines

When using `mode='replace'` on a standard Gen2 dataflow, the original is deleted and a new CI/CD dataflow is created with a **new ID**. Any pipelines referencing the old dataflow ID need to be updated. The full workflow is:

1. `change_data_destination(mode='replace')` — replace the dataflow
2. `find_pipelines_by_dataflow` — find pipelines still referencing the old ID
3. `replace_dataflow_id_in_pipeline` — update each pipeline to point to the new ID

### Dataflow - Get Data Destinations

In [ ]:
# Get data destinations for each table in a dataflow
workspace_id = 'your-workspace-id'
dataflow_id = 'your-dataflow-id'

result = dataflow.get_data_destinations(workspace_id=workspace_id, dataflow_id=dataflow_id)

if result['message'] == 'Success':
    df_destinations = pd.DataFrame(result['content'])
    df_destinations
else:
    print(f"Error: {result['message']}")

In [ ]:
# Step 1: Change data destination (replace mode — deletes original, creates new CI/CD)
workspace_id = 'your-workspace-id'
old_dataflow_id = 'your-dataflow-id'

result = dataflow.change_data_destination(
    workspace_id=workspace_id,
    dataflow_id=old_dataflow_id,
    destination_type='Warehouse',
    destination_workspace_id='your-destination-workspace-id',
    destination_item_id='your-warehouse-id',
    mode='replace'
)

if result['message'] == 'Success':
    new_dataflow_id = result['content']['id']
    print(f"New dataflow created: {new_dataflow_id}")
else:
    print(f"Error: {result['message']}")

In [ ]:
# Step 2: Find pipelines that still reference the old dataflow ID
matches = pipeline.find_pipelines_by_dataflow(workspace_id=workspace_id, dataflow_id=old_dataflow_id)

if matches['message'] == 'Success':
    print(f"Found {len(matches['content'])} pipeline(s) to update:")
    for m in matches['content']:
        print(f"  {m['pipeline_name']} — activities: {', '.join(m['activities'])}")
else:
    print(f"Error: {matches['message']}")

In [ ]:
# Step 3: Update each pipeline to point to the new dataflow ID
for m in matches['content']:
    result = pipeline.replace_dataflow_id_in_pipeline(
        workspace_id=workspace_id,
        pipeline_id=m['pipeline_id'],
        old_dataflow_id=old_dataflow_id,
        new_dataflow_id=new_dataflow_id
    )
    print(f"  {m['pipeline_name']}: {result['message']}")

### Database - Write DataFrame

In [ ]:
# Write a DataFrame to a SQL table
df = pd.DataFrame({'col1': [1, 2, 3], 'col2': ['a', 'b', 'c']})

result = db.write_dataframe(
    df=df,
    table_name='my_table',
    schema='dbo',
    if_exists='append',  # Options: 'fail', 'replace', 'append', 'delete_rows'
    chunksize=10000
)
print(result)